# P7-2 · Vision-language model

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
# Cần GPU và mạng: tải Qwen2.5-VL-3B (khoảng 7 GB) và ảnh ghe_sofa.jpg của bạn, chạy trên Colab
from transformers import pipeline
vlm = pipeline("image-text-to-text", model="Qwen/Qwen2.5-VL-3B-Instruct", device_map="auto")
messages = [{"role": "user", "content": [
    {"type": "image", "url": "ghe_sofa.jpg"},
    {"type": "text", "text": "Đây là sản phẩm gì, màu gì, chất liệu gì? Trả lời ngắn gọn."},
]}]
out = vlm(text=messages, max_new_tokens=80)
print(out[0]["generated_text"][-1]["content"])

## Bài tập 4: Hỏi đáp ảnh với VLM mã nguồn mở {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần GPU, mạng và ảnh san_pham.jpg của bạn: chạy trên Colab
from transformers import pipeline

vlm = pipeline("image-text-to-text", model="Qwen/Qwen2.5-VL-3B-Instruct", device_map="auto")
messages = [{"role": "user", "content": [
    {"type": "image", "url": "san_pham.jpg"},
    {"type": "text", "text": 'Đây là sản phẩm gì, màu gì, chất liệu gì? Chỉ trả JSON {"loai": "", "mau": "", "chat_lieu": ""}.'},
]}]
out = vlm(text=messages, max_new_tokens=120)
print(out[0]["generated_text"][-1]["content"])

## Bài tập 5: Ghép token ảnh vào chuỗi của LLM {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
vis = torch.randn(1, 576, 1024)                # đầu ra vision encoder (CLIP ViT-L/14 @336)
sys_emb, q_emb, ans_emb = torch.randn(1, 12, 2048), torch.randn(1, 20, 2048), torch.randn(1, 15, 2048)
ans_ids = torch.randint(0, 32000, (1, 15))
# TODO: projector; img_tok; seq (ghép 4 phần); labels (-100 trừ 15 vị trí cuối)

In [ ]:
projector = nn.Sequential(nn.Linear(1024, 2048), nn.GELU(), nn.Linear(2048, 2048))
img_tok = projector(vis)
seq = torch.cat([sys_emb, img_tok, q_emb, ans_emb], dim=1)
labels = torch.full((1, seq.shape[1]), -100)
labels[0, -15:] = ans_ids[0]
print(f"projector {sum(p.numel() for p in projector.parameters()):,} tham số | chuỗi {seq.shape[1]} token, "
      f"trong đó ảnh {img_tok.shape[1]} | tính loss trên {(labels != -100).sum().item()} vị trí")

projector 6,295,552 tham số | chuỗi 623 token, trong đó ảnh 576 | tính loss trên 15 vị trí


## Bài tập 6: Ngân sách token ảnh theo độ phân giải {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import math
CANH, TOK, MAX_O = 336, 576, 12

def so_token(w, h):
    # TODO: 1 ô thì 576; nhiều ô: số ô (tối đa 12, thu nhỏ nếu vượt) × 576 + 576 cho ảnh thu nhỏ
    return 0

In [ ]:
def so_token(w, h):
    o = math.ceil(w / CANH) * math.ceil(h / CANH)
    if o == 1:
        return TOK
    s = 1.0
    while o > MAX_O:
        s -= 0.05
        o = math.ceil(w * s / CANH) * math.ceil(h * s / CANH)
    return o * TOK + TOK

for ten, (w, h) in [("sản phẩm", (336, 336)), ("hóa đơn", (1008, 1344)), ("trang tài liệu", (2016, 2688))]:
    t = so_token(w, h)
    print(f"{ten:15s}: {t:5d} token ≈ {t * 3 / 1e6 * 25000:,.0f} đồng (3 USD/triệu token, 25.000đ/USD)")

sản phẩm       :   576 token ≈ 43 đồng (3 USD/triệu token, 25.000đ/USD)
hóa đơn        :  7488 token ≈ 562 đồng (3 USD/triệu token, 25.000đ/USD)
trang tài liệu :  7488 token ≈ 562 đồng (3 USD/triệu token, 25.000đ/USD)


## Bài tập 7: Kiểm tra chéo kết quả VLM đọc hóa đơn {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import re
hoa_don = [
    {"so": "HD01", "ngay": "05/11/2026", "dong": [{"sl": 2, "dg": 1500000, "tt": 3000000}, {"sl": 1, "dg": 450000, "tt": 450000}], "tong_cong": 3450000},
    {"so": "HD02", "ngay": "06/11/2026", "dong": [{"sl": 3, "dg": 250000, "tt": 750000}], "tong_cong": 780000},
    {"so": "HD03", "ngay": "2026-11-07", "dong": [{"sl": 1, "dg": 320000, "tt": 320000}], "tong_cong": 320000},
    {"so": "HD04", "ngay": "08/11/2026", "dong": [{"sl": 2, "dg": 280000, "tt": 650000}, {"sl": 1, "dg": 90000, "tt": 90000}], "tong_cong": 740000},
]

def kiem_tra(hd):
    # TODO: trả về list lỗi
    return []

In [ ]:
def kiem_tra(hd):
    loi = []
    for i, d in enumerate(hd["dong"], 1):
        if d["sl"] * d["dg"] != d["tt"]:
            loi.append(f"dòng {i}: {d['sl']} × {d['dg']:,} = {d['sl'] * d['dg']:,} ≠ {d['tt']:,}")
    if sum(d["tt"] for d in hd["dong"]) != hd["tong_cong"]:
        loi.append(f"tổng các dòng {sum(d['tt'] for d in hd['dong']):,} ≠ tổng cộng {hd['tong_cong']:,}")
    if not re.fullmatch(r"\d{2}/\d{2}/\d{4}", hd["ngay"]):
        loi.append(f"ngày sai định dạng: {hd['ngay']}")
    return loi

for hd in hoa_don:
    l = kiem_tra(hd)
    print(hd["so"], "OK" if not l else "CẦN NGƯỜI KIỂM TRA: " + "; ".join(l))

HD01 OK
HD02 CẦN NGƯỜI KIỂM TRA: tổng các dòng 750,000 ≠ tổng cộng 780,000
HD03 CẦN NGƯỜI KIỂM TRA: ngày sai định dạng: 2026-11-07
HD04 CẦN NGƯỜI KIỂM TRA: dòng 1: 2 × 280,000 = 560,000 ≠ 650,000
